# Day 20 — Final Release Verification

Day 20 adds no new ML. Its job is to make the repository something a stranger can trust, so this notebook
**re-checks the release against itself**: every document exists and links resolve; the numbers quoted in the
README and reports match the artifacts they came from; the code behaves the way the docs say; the API still
matches its published schema; the tests, frontend build and lint run; the assets are intact; and the git
repository is clean and free of obvious secrets.

Each check is recorded as **PASS**, **WARN** (known and acceptable, with the reason), **FAIL** (a real
problem — the notebook stops), or **INFO** (something it can report but not judge).

**What this notebook cannot tell you** (listed again at the end): whether the prose is *right* (only that its
numbers match), whether the Docker image builds (Docker is not installed here), whether CI passes on GitHub,
or how the UI feels to a person. Rendering the UI was done separately with headless Chrome (Phase G); Mermaid
diagrams were validated with the Mermaid parser (Phase C).

Run the heavy test lane too with `RUN_HEAVY=1` (about 5½ minutes).

In [1]:
import os, sys, re, json, glob, subprocess, shutil, time
os.environ["OMP_NUM_THREADS"] = "1"           # before any faiss/torch import (TestClient loads both)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
RUN_HEAVY = os.environ.get("RUN_HEAVY", "0") == "1"
RESULTS = []

def check(area, name, ok, detail="", level="FAIL"):
    status = "PASS" if ok else level
    RESULTS.append((area, name, status, str(detail)))
    print(f"[{status:4}] {area}: {name}" + (f"  — {detail}" if detail != "" else ""))
    return bool(ok)

def note(area, name, detail=""):
    RESULTS.append((area, name, "INFO", str(detail)))
    print(f"[INFO] {area}: {name}" + (f"  — {detail}" if detail != "" else ""))

def read(rel):
    return (ROOT / rel).read_text(encoding="utf-8")

print("repo:", ROOT, "| heavy test lane:", "ON" if RUN_HEAVY else "off (set RUN_HEAVY=1)")

repo: /Users/Yash/AI-Ventutre-Intelligence-Platform | heavy test lane: ON


## 1 — The release files exist and are not empty

At the start of Day 20 the root `README.md` and `LICENSE` were **0-byte files**. This confirms the fix held
and that every document in the release set is present.

In [2]:
REQUIRED = [
    "README.md", "LICENSE", "CHANGELOG.md", "CONTRIBUTING.md", "ruff.toml", "requirements.txt",
    "docs/PROJECT_OVERVIEW.md", "docs/SYSTEM_ARCHITECTURE.md", "docs/INSTALLATION.md", "docs/DEPLOYMENT.md",
    "docs/API_DOCUMENTATION.md", "docs/PERFORMANCE_REPORT.md", "docs/DEMO_GUIDE.md",
    "docs/PORTFOLIO_AND_RESUME.md", "docs/INTERVIEW_GUIDE.md", "docs/RESEARCH_ROADMAP.md", "docs/assets/README.md",
]
for f in REQUIRED:
    p = ROOT / f
    size = p.stat().st_size if p.exists() else -1
    check("files", f, p.is_file() and size > 200, f"{size:,} bytes")

lic = read("LICENSE")
check("files", "LICENSE is MIT, holder yashraj-a11y", "MIT License" in lic and "yashraj-a11y" in lic)

readme = read("README.md")
SECTIONS = ["Motivation", "Features", "Architecture", "Quick start", "Dataset", "ML pipeline", "AI pipeline",
            "API usage", "Screenshots", "Deployment", "Results", "Limitations", "Future work", "Author", "License"]
missing = [s for s in SECTIONS if not re.search(rf"^## {re.escape(s)}\b", readme, re.M)]
check("files", "README has every planned section", not missing, f"missing: {missing}" if missing else f"{len(SECTIONS)} sections")

[PASS] files: README.md  — 9,094 bytes
[PASS] files: LICENSE  — 1,069 bytes
[PASS] files: CHANGELOG.md  — 4,241 bytes
[PASS] files: CONTRIBUTING.md  — 1,799 bytes
[PASS] files: ruff.toml  — 230 bytes
[PASS] files: requirements.txt  — 3,868 bytes
[PASS] files: docs/PROJECT_OVERVIEW.md  — 4,594 bytes
[PASS] files: docs/SYSTEM_ARCHITECTURE.md  — 16,816 bytes
[PASS] files: docs/INSTALLATION.md  — 3,808 bytes
[PASS] files: docs/DEPLOYMENT.md  — 2,558 bytes
[PASS] files: docs/API_DOCUMENTATION.md  — 39,411 bytes
[PASS] files: docs/PERFORMANCE_REPORT.md  — 15,737 bytes
[PASS] files: docs/DEMO_GUIDE.md  — 20,399 bytes
[PASS] files: docs/PORTFOLIO_AND_RESUME.md  — 17,504 bytes
[PASS] files: docs/INTERVIEW_GUIDE.md  — 24,199 bytes
[PASS] files: docs/RESEARCH_ROADMAP.md  — 12,471 bytes
[PASS] files: docs/assets/README.md  — 5,703 bytes
[PASS] files: LICENSE is MIT, holder yashraj-a11y
[PASS] files: README has every planned section  — 15 sections


True

## 2 — Documentation integrity

Every relative link and image in the release documents must resolve, and the rendered diagrams must match the
Mermaid sources in the architecture document. (Mermaid *syntax* is validated with the real parser in Phase C;
Python cannot run it, so that is not repeated here.)

In [3]:
RELEASE_DOCS = ["README.md", "CHANGELOG.md", "CONTRIBUTING.md"] + sorted(
    p.as_posix() for p in (ROOT / "docs").glob("*.md") if not p.name.startswith("day")) + ["docs/assets/README.md"]

broken = []
for f in RELEASE_DOCS:
    base = (ROOT / f).parent
    for m in re.finditer(r'(?:\]\(|src=")([^)"#\s]+)', read(f)):
        t = m.group(1)
        if t.startswith(("http", "mailto")):
            continue
        if not (base / t).resolve().exists():
            broken.append((f, t))
check("docs", f"links and images resolve across {len(RELEASE_DOCS)} release documents", not broken, broken or "0 broken")

n_mmd = len(re.findall(r"```mermaid", read("docs/SYSTEM_ARCHITECTURE.md")))
n_png = len(list((ROOT / "docs/assets/diagrams").glob("*.png")))
n_svg = len(list((ROOT / "docs/assets/diagrams").glob("*.svg")))
check("docs", "rendered diagrams match the Mermaid blocks", n_mmd == n_png == n_svg, f"{n_mmd} mermaid / {n_png} png / {n_svg} svg")

# placeholders: the two that are meant to remain are the user's own (demo video, name)
todo = {}
for f in RELEASE_DOCS:
    for m in re.finditer(r"\[(demo video URL|your name|repo URL)\]|\bTODO\b|\bFIXME\b|<your-fork-url>", read(f)):
        todo.setdefault(f, set()).add(m.group(0))
unexpected = {f: v for f, v in todo.items() if v - {"[demo video URL]", "[your name]"}}
check("docs", "no stray TODO / unfilled placeholders", not unexpected, unexpected or "none")
check("docs", "placeholders still waiting on the owner", not todo, sorted({x for v in todo.values() for x in v}), level="WARN")

[PASS] docs: links and images resolve across 23 release documents  — 0 broken
[PASS] docs: rendered diagrams match the Mermaid blocks  — 11 mermaid / 11 png / 11 svg
[PASS] docs: no stray TODO / unfilled placeholders  — none
[WARN] docs: placeholders still waiting on the owner  — ['[demo video URL]', '[your name]']


False

## 3 — The numbers in the docs match the artifacts

The performance report, README and resume kit quote metrics. Here each is **re-read from its source artifact**
and must appear, formatted the same way, in the documents that cite it. A mismatch means a doc went stale.

In [4]:
mv   = json.loads(read("artifacts/testing/model_validation.json"))
meta = json.loads(read("artifacts/models/model_metadata.json"))
logr = json.loads(read("artifacts/models/logreg_results.json"))
feat = json.loads(read("artifacts/feature_pipeline/feature_metadata.json"))
rv   = json.loads(read("artifacts/testing/retrieval_validation.json"))
bm   = json.loads(read("artifacts/testing/benchmark.json"))

def appears(needle, files):
    return [f for f in files if needle in read(f)]

CLAIMS = [
    ("test ROC-AUC",            f"{mv['test_metrics']['roc_auc']:.4f}",   ["README.md", "docs/PERFORMANCE_REPORT.md"]),
    ("test PR-AUC",             f"{mv['test_metrics']['pr_auc']:.4f}",    ["README.md", "docs/PERFORMANCE_REPORT.md"]),
    ("test F1",                 f"{mv['test_metrics']['f1']:.4f}",        ["docs/PERFORMANCE_REPORT.md"]),
    ("test recall",             f"{mv['test_metrics']['recall']:.4f}",    ["docs/PERFORMANCE_REPORT.md"]),
    ("test precision",          f"{mv['test_metrics']['precision']:.4f}", ["docs/PERFORMANCE_REPORT.md"]),
    ("test Brier",              f"{mv['test_metrics']['brier']:.4f}",     ["README.md", "docs/PERFORMANCE_REPORT.md"]),
    ("test F1",                 f"{mv['test_metrics']['f1']:.4f}",        ["README.md"]),
    ("decision threshold",      f"{meta['decision_threshold']}",          ["README.md", "docs/PERFORMANCE_REPORT.md", "docs/INTERVIEW_GUIDE.md"]),
    ("calibrated val Brier",    f"{meta['validation_metrics']['calibrated_val']['brier']:.4f}", ["docs/PERFORMANCE_REPORT.md"]),
    ("baseline val PR-AUC",     f"{logr['logreg_l2']['metrics']['pr_auc']:.4f}", ["README.md", "docs/PERFORMANCE_REPORT.md"]),
    ("baseline val ROC-AUC",    f"{logr['logreg_l2']['metrics']['roc_auc']:.4f}", ["docs/PERFORMANCE_REPORT.md"]),
    ("feature count",           f"{feat['n_features_out']}",              ["docs/PERFORMANCE_REPORT.md", "docs/INTERVIEW_GUIDE.md", "docs/PORTFOLIO_AND_RESUME.md"]),
    ("retrieval category lift", f"{rv['category_consistency']['k5']['lift']:.1f}×", ["docs/PERFORMANCE_REPORT.md", "docs/INTERVIEW_GUIDE.md"]),
    ("predict p50 latency",     f"{round(bm['latency_ms']['predict']['p50'])} ms", ["docs/INTERVIEW_GUIDE.md", "docs/PORTFOLIO_AND_RESUME.md"]),
]
rows = []
for name, needle, files in CLAIMS:
    found = appears(needle, files)
    ok = len(found) == len(files)
    check("claims", f"{name} = {needle}", ok, "quoted in " + ", ".join(Path(f).name for f in found) if ok
          else f"NOT found in {[Path(f).name for f in set(files) - set(found)]}")
    rows.append((name, needle, len(found), len(files)))

tfidf_cfg = re.search(r"max_features:\s*(\d+)", read("configs/preprocessing.yaml")).group(1)
check("claims", "TF-IDF is 800 features in the config (not the stale 2,500)", tfidf_cfg == "800", f"max_features={tfidf_cfg}")
lines2500 = [(f, i, l.strip()) for f in RELEASE_DOCS for i, l in enumerate(read(f).splitlines(), 1) if re.search(r"\b2,?500\b", l)]
bad2500 = [x for x in lines2500 if not re.search(r"stale|not|never|❌|instead|draft|wrong|isn't|NOT", x[2])]
check("claims", "every mention of 2,500 features in the docs is a correction", not bad2500,
      bad2500 or f"{len(lines2500)} mention(s), all negated")

[PASS] claims: test ROC-AUC = 0.7763  — quoted in README.md, PERFORMANCE_REPORT.md
[PASS] claims: test PR-AUC = 0.6883  — quoted in README.md, PERFORMANCE_REPORT.md
[PASS] claims: test F1 = 0.6664  — quoted in PERFORMANCE_REPORT.md
[PASS] claims: test recall = 0.8131  — quoted in PERFORMANCE_REPORT.md
[PASS] claims: test precision = 0.5646  — quoted in PERFORMANCE_REPORT.md
[PASS] claims: test Brier = 0.1873  — quoted in README.md, PERFORMANCE_REPORT.md
[PASS] claims: test F1 = 0.6664  — quoted in README.md
[PASS] claims: decision threshold = 0.329  — quoted in README.md, PERFORMANCE_REPORT.md, INTERVIEW_GUIDE.md
[PASS] claims: calibrated val Brier = 0.1872  — quoted in PERFORMANCE_REPORT.md
[PASS] claims: baseline val PR-AUC = 0.6264  — quoted in README.md, PERFORMANCE_REPORT.md
[PASS] claims: baseline val ROC-AUC = 0.7269  — quoted in PERFORMANCE_REPORT.md
[PASS] claims: feature count = 865  — quoted in PERFORMANCE_REPORT.md, INTERVIEW_GUIDE.md, PORTFOLIO_AND_RESUME.md
[PASS] claims:

True

## 4 — The code behaves the way the docs say

Claims about structure and honesty are checked against the code itself: how many layers and agents exist, that
the six unsupported scores are really deferred (`None`), that no neural network was ever built, and that any
mention of one in the docs is a negation.

In [5]:
# layers: every layer 2..13 is labelled in a source docstring (layer 1 is the preprocessing/feature stage)
layer_ids = set()
for p in (ROOT / "src").rglob("*.py"):
    layer_ids |= {int(x) for x in re.findall(r"Layer (\d{1,2})\b", p.read_text(encoding="utf-8", errors="ignore")[:1500])}
check("code", "layers 2–13 are each labelled in source", set(range(2, 14)) <= layer_ids, f"found {sorted(layer_ids)}")

n_agents = len(list((ROOT / "src/agents").glob("*_agent.py")))
check("code", "six agents (market, founder, financial, competition, risk, investment)", n_agents == 6, f"{n_agents} agent modules")
check("code", "a consensus engine exists", (ROOT / "src/agents/consensus.py").is_file())

from src.models.venture_scores import VentureScoreGenerator
vs = VentureScoreGenerator().score_campaign(0.40)
deferred = sorted(k for k, v in vs.items() if isinstance(v, dict) and v.get("status") == "deferred")
FIVE = {"business_model", "founder_strength", "growth_potential", "market_fit", "scalability"}
check("code", "exactly five scores are deferred with value None (never invented)",
      set(deferred) == FIVE and all(vs[k]["value"] is None for k in deferred), deferred)
inv = json.loads(next((ROOT / "docs/assets/sample_reports").glob("investor_due_diligence_report__*.json")).read_text())
memo = json.loads(next((ROOT / "docs/assets/sample_reports").glob("investment_memo__*.json")).read_text())
check("code", "market SIZE (TAM/SAM) is deferred — listed in the report's deferred appendix",
      "market_size_tam" in inv["deferred_appendix"]["items"], "market_size_tam present")
check("code", "…while a category-level market-opportunity signal IS computed (so 'Market Opportunity' is not a deferred score)",
      any(sec["id"] == "market_opportunity" for sec in memo["sections"]), "memo has a 'Market Opportunity' section")
ui = read("frontend/src/components/Dashboard/DeferredScores.tsx")
check("code", "the UI's six deferred cards = the five scores + market size (no 'Market Opportunity' card)",
      '"Market Size (TAM / SAM)"' in ui and '"Market Opportunity"' not in ui, "DeferredScores.tsx")
real = sorted(k for k, v in vs.items() if isinstance(v, dict) and v.get("status") != "deferred")
note("code", "scores the platform does compute", real)

nn_hits = [p.as_posix() for p in (ROOT / "src").rglob("*.py")
           if re.search(r"MLPClassifier|torch\.nn\b|keras|tensorflow", p.read_text(encoding="utf-8", errors="ignore"))]
check("code", "no neural-network model is implemented (torch is used for SBERT only)", not nn_hits, nn_hits or "none")

claims_nn = []
NEG = re.compile(r"\b(not|never|no |deferred|didn't|isn't|without|draft|none|n't)\b", re.I)
for f in RELEASE_DOCS:
    for i, line in enumerate(read(f).splitlines(), 1):
        if re.search(r"(?<!graph )neural[- ]network", line, re.I) and not NEG.search(line):
            claims_nn.append(f"{f}:{i}: {line.strip()[:100]}")
check("code", "every mention of a neural network in the docs is a negation / deferral", not claims_nn, claims_nn or "all negated", level="WARN")

[PASS] code: layers 2–13 are each labelled in source  — found [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
[PASS] code: six agents (market, founder, financial, competition, risk, investment)  — 6 agent modules
[PASS] code: a consensus engine exists


[PASS] code: exactly five scores are deferred with value None (never invented)  — ['business_model', 'founder_strength', 'growth_potential', 'market_fit', 'scalability']
[PASS] code: market SIZE (TAM/SAM) is deferred — listed in the report's deferred appendix  — market_size_tam present
[PASS] code: …while a category-level market-opportunity signal IS computed (so 'Market Opportunity' is not a deferred score)  — memo has a 'Market Opportunity' section
[PASS] code: the UI's six deferred cards = the five scores + market size (no 'Market Opportunity' card)  — DeferredScores.tsx
[INFO] code: scores the platform does compute  — ['failure_risk', 'funding_probability', 'investment_readiness', 'investment_score', 'startup_score']
[PASS] code: no neural-network model is implemented (torch is used for SBERT only)  — none
[WARN] code: every mention of a neural network in the docs is a negation / deferral  — ['/Users/Yash/AI-Ventutre-Intelligence-Platform/docs/DEMO_GUIDE.md:260: **Q6. "Where\'s the

False

## 5 — The API matches its published contract

Boots the real application, compares its live OpenAPI schema with the committed `docs/assets/openapi.json`
(so the API documentation cannot have drifted), calls every analysis endpoint with the demo campaign, and
confirms the honesty rule: an invented field is rejected with `422`.

In [6]:
from fastapi.testclient import TestClient
from src.api.main import app

campaign = json.loads(read("docs/assets/campaign_A.json"))
spec_live = app.openapi()
spec_doc = json.loads(read("docs/assets/openapi.json"))
check("api", "docs/assets/openapi.json equals the live schema", spec_live == spec_doc)

def required_keys(path):
    sch = spec_live["paths"][path]["post"]["responses"]["200"]["content"]["application/json"]["schema"]
    name = sch["$ref"].rsplit("/", 1)[-1]
    return set(spec_live["components"]["schemas"][name].get("required", []))

POSTS = ["/predict", "/explain", "/retrieve?k=3", "/counterfactual", "/scenario", "/recommend",
         "/analyze", "/report/founder", "/report/investor"]
timing = []
with TestClient(app, raise_server_exceptions=False) as client:
    check("api", "/ready reports the stack loaded", client.get("/ready").json() == {"status": "ready"})
    for p in POSTS:
        t0 = time.time()
        r = client.post(p, json=campaign)
        dt = time.time() - t0
        base = p.split("?")[0]
        keys = set(r.json()) if r.status_code == 200 else set()
        ok = r.status_code == 200 and required_keys(base) <= keys
        check("api", f"POST {p}", ok, f"HTTP {r.status_code}, {dt:.2f}s, required keys present")
        timing.append((p, r.status_code, round(dt, 2)))
    bad = client.post("/predict", json={**campaign, "founder_experience_years": 7})
    check("api", "an invented field is rejected with 422", bad.status_code == 422 and
          bad.json()["detail"][0]["type"] == "extra_forbidden", f"HTTP {bad.status_code}")
    bad2 = client.post("/predict", json={**campaign, "deadline": "2017-03-01 10:00:00"})
    check("api", "deadline before launch is rejected with 422", bad2.status_code == 422)
    exp = client.post("/explain", json=campaign).json()
    additive = abs((exp["base_value"] + sum(c["contribution"] for c in exp["top_positive"] + exp["top_negative"])) - exp["margin"])
    note("api", "/explain margin vs base+listed contributions (rest is the unlisted tail)", f"gap {additive:.3f}")
    pr = client.post("/predict", json=campaign).json()
    check("api", "demo campaign A still predicts ≈ 40% (documented in the demo guide)",
          abs(pr["funding_probability"] - 0.4021) < 0.005, f"{pr['funding_probability']}")
pd.DataFrame(timing, columns=["endpoint", "status", "seconds"])

[PASS] api: docs/assets/openapi.json equals the live schema
2026-11-03 02:04:43 | avip.api                         | INFO     | API starting — OMP_NUM_THREADS=1, log_file=None, level=INFO


/Users/Yash/AI-Ventutre-Intelligence-Platform/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


2026-11-03 02:04:43 | avip.core.registry               | INFO     | PredictionEngine ready — model=calibrated_model.joblib, threshold=0.329.


2026-11-03 02:04:44 | avip.core.registry               | INFO     | Item frame: 265171 campaigns, 13 item features, base funding rate 0.4038.


2026-11-03 02:04:44 | avip.core.registry               | INFO     | CounterfactualPipeline ready (warm model + item frame).


2026-11-03 02:04:44 | avip.core.registry               | INFO     | Loaded LGBMClassifier (865 features).


... (253 outputs left out)

[INFO] api: /explain margin vs base+listed contributions (rest is the unlisted tail)  — gap 0.208
2026-11-03 02:05:13 | avip.preprocessing.cleaning      | INFO     | TextCleaner: wrote cleaned titles to 'name_clean' (raw 'name' preserved).


2026-11-03 02:05:13 | avip.api                         | INFO     | [932aadb1] POST /predict -> 200 (12ms)


[PASS] api: demo campaign A still predicts ≈ 40% (documented in the demo guide)  — 0.4021
2026-11-03 02:05:13 | avip.api                         | INFO     | API shutting down.


,endpoint,status,seconds
0,/predict,200,1.35
1,/explain,200,0.05
2,/retrieve?k=3,200,9.51
3,/counterfactual,200,0.36
4,/scenario,200,0.06
5,/recommend,200,1.16
6,/analyze,200,3.93
7,/report/founder,200,2.40
8,/report/investor,200,3.43


## 6 — The test suite

Counts the tests, runs the **fast lane** (the 53 tests CI runs, no artifacts needed), and — with
`RUN_HEAVY=1` — the **full 95** (needs the git-ignored models, data and FAISS index). The heavy lane rewrites
`artifacts/testing/*.json` as a side effect; those regenerated files are restored afterwards so the repo stays
clean.

In [7]:
def run(cmd, timeout=None):
    env = {**os.environ, "OMP_NUM_THREADS": "1"}
    return subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT, env=env, timeout=timeout)

PY = [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"]
col = run(PY + ["--collect-only"])
m = re.search(r"(\d+) tests? collected", col.stdout)
n_collected = int(m.group(1)) if m else 0
check("tests", "95 tests are collected (the number the docs quote)", n_collected == 95, f"{n_collected} collected")

t0 = time.time()
fast = run(PY + ["-m", "not heavy"])
m = re.search(r"(\d+) passed", fast.stdout)
n_fast = int(m.group(1)) if m else 0
check("tests", "fast lane passes (53 tests, what CI runs)", fast.returncode == 0 and n_fast == 53,
      f"{n_fast} passed in {time.time() - t0:.1f}s")

if RUN_HEAVY:
    t0 = time.time()
    full = run(PY)
    m = re.search(r"(\d+) passed", full.stdout)
    n_full = int(m.group(1)) if m else 0
    failed = re.search(r"(\d+) failed", full.stdout)
    check("tests", "FULL suite passes (95 tests)", full.returncode == 0 and n_full == 95 and not failed,
          f"{n_full} passed in {time.time() - t0:.0f}s" + (f", {failed.group(1)} failed" if failed else ""))
    dirty = run(["git", "status", "--porcelain", "artifacts/testing"]).stdout.strip()
    if dirty:
        run(["git", "checkout", "--", "artifacts/testing"])
        note("tests", "restored artifacts/testing/*.json regenerated by the heavy lane", f"{len(dirty.splitlines())} files")
else:
    note("tests", "heavy lane (42 tests) not run in this execution", "set RUN_HEAVY=1; last full run in this release: 95 passed")

[PASS] tests: 95 tests are collected (the number the docs quote)  — 95 collected


[PASS] tests: fast lane passes (53 tests, what CI runs)  — 53 passed in 4.5s


[PASS] tests: FULL suite passes (95 tests)  — 95 passed in 1764s
[INFO] tests: restored artifacts/testing/*.json regenerated by the heavy lane  — 6 files


## 7 — Frontend build and lint

`npm run build` runs `tsc -b` (the typecheck) and then the Vite bundle, so a green build means the types
compile. Lint is `ruff` over `src/` and `tests/` with the repo's conservative `ruff.toml` (real errors only).

In [8]:
env = {**os.environ, "PATH": "/opt/homebrew/bin:/usr/local/bin:" + os.environ.get("PATH", "")}
npm = shutil.which("npm", path=env["PATH"])
if npm:
    t0 = time.time()
    r = subprocess.run([npm, "run", "build"], cwd=ROOT / "frontend", capture_output=True, text=True, env=env)
    ok = r.returncode == 0 and (ROOT / "frontend/dist/index.html").is_file()
    js = sorted((ROOT / "frontend/dist/assets").glob("*.js"), key=lambda p: -p.stat().st_size)
    check("frontend", "npm run build (typecheck + bundle) succeeds", ok,
          f"{time.time() - t0:.1f}s; largest chunk {js[0].name} {js[0].stat().st_size // 1024} KB" if ok else r.stderr[-300:])
else:
    note("frontend", "npm not found — frontend build not run")

r = run([sys.executable, "-m", "ruff", "check", "src", "tests", "--output-format", "concise"])
m = re.search(r"Found (\d+) error", r.stdout + r.stderr)
n_lint = int(m.group(1)) if m else 0
check("lint", "ruff: only the 3 known unused-variable warnings remain", n_lint <= 3,
      f"{n_lint} finding(s) (3 are left on purpose: removing them could change behaviour)", level="FAIL")

[PASS] frontend: npm run build (typecheck + bundle) succeeds  — 1.8s; largest chunk ExplanationPanel-BSRiaPBM.js 348 KB
[PASS] lint: ruff: only the 3 known unused-variable warnings remain  — 3 finding(s) (3 are left on purpose: removing them could change behaviour)


True

## 8 — Portfolio assets are intact

Counts each asset group and confirms every image is a valid PNG of a sensible size.

In [9]:
from PIL import Image
A = ROOT / "docs/assets"
GROUPS = {
    "screenshots": (A / "screenshots", "*.png", 19),
    "diagrams (png)": (A / "diagrams", "*.png", 11),
    "diagrams (svg)": (A / "diagrams", "*.svg", 11),
    "charts": (A / "charts", "*.png", 11),
    "sample reports": (A / "sample_reports", "*", 12),
    "API examples": (A / "examples", "*.json", 7),
}
for name, (d, pat, expect) in GROUPS.items():
    n = len([p for p in d.glob(pat) if p.is_file()])
    check("assets", f"{name}: {expect} files", n == expect, f"{n} found")

bad_img = []
for p in list(A.rglob("*.png")):
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            if min(im.size) < 100:
                bad_img.append((p.name, im.size))
    except Exception as e:
        bad_img.append((p.name, str(e)[:40]))
check("assets", f"all {len(list(A.rglob('*.png')))} PNGs open and are not tiny", not bad_img, bad_img or "ok")
check("assets", "social preview card exists (1280×640)", Image.open(A / "social-preview.png").size == (1280, 640))
total_mb = sum(p.stat().st_size for p in A.rglob("*") if p.is_file()) / 1e6
note("assets", "total size of docs/assets", f"{total_mb:.1f} MB")

[PASS] assets: screenshots: 19 files  — 19 found
[PASS] assets: diagrams (png): 11 files  — 11 found
[PASS] assets: diagrams (svg): 11 files  — 11 found
[PASS] assets: charts: 11 files  — 11 found
[PASS] assets: sample reports: 12 files  — 12 found
[PASS] assets: API examples: 7 files  — 7 found
[PASS] assets: all 42 PNGs open and are not tiny  — ok
[PASS] assets: social preview card exists (1280×640)
[INFO] assets: total size of docs/assets  — 7.3 MB


## 9 — Repository hygiene

Git state (local only — this notebook never contacts GitHub), nothing tracked that GitHub would reject, no
obvious secrets, and the environment file excluded.

In [10]:
def sh(*a):
    return subprocess.run(a, capture_output=True, text=True, cwd=ROOT).stdout.strip()

branch = sh("git", "branch", "--show-current")
dirty = [l for l in sh("git", "status", "--porcelain").splitlines() if "20_Final_Release" not in l]
check("git", f"working tree clean on '{branch}' (ignoring this notebook)", not dirty, dirty[:5] or "clean")
note("git", "last commit", sh("git", "log", "-1", "--format=%h %an | %s"))

tracked = sh("git", "ls-files", "-z").split("\0")
sizes = sorted(((os.path.getsize(ROOT / f), f) for f in tracked if f and (ROOT / f).is_file()), reverse=True)
check("git", "no tracked file exceeds GitHub's 100 MB limit", sizes[0][0] < 100 * 1024 * 1024,
      f"largest: {sizes[0][1]} ({sizes[0][0] / 1e6:.1f} MB)")

check("git", ".env is not tracked and is git-ignored",
      not sh("git", "ls-files", ".env") and ".env" in read(".gitignore").splitlines())
ign = sh("git", "check-ignore", "frontend/node_modules", "frontend/dist", "mlruns", "logs", "artifacts/vector_db/faiss.index")
check("git", "node_modules, dist, mlruns, logs and the FAISS index are ignored", len(ign.splitlines()) == 5, ign.replace("\n", ", "))

PATTERNS = {"AWS key": r"AKIA[0-9A-Z]{16}", "GitHub token": r"gh[pousr]_[A-Za-z0-9]{36,}",
            "OpenAI-style key": r"sk-[A-Za-z0-9]{32,}", "private key": r"-----BEGIN (RSA |EC |OPENSSH )?PRIVATE KEY-----"}
hits = []
for f in tracked:
    p = ROOT / f
    if not f or not p.is_file() or p.stat().st_size > 2_000_000 or p.suffix in {".png", ".npz", ".npy", ".joblib", ".pkl", ".svg"}:
        continue
    try:
        txt = p.read_text(encoding="utf-8")
    except Exception:
        continue
    for label, pat in PATTERNS.items():
        if re.search(pat, txt):
            hits.append((f, label))
check("git", "no obvious secrets in tracked text files", not hits, hits or f"scanned {len(tracked)} tracked paths")

ahead = sh("git", "rev-list", "--count", "origin/main..HEAD")
check("git", "all commits are pushed to origin/main", ahead in ("", "0"),
      f"{ahead} local commit(s) not yet on GitHub (pushing is the owner's decision)", level="WARN")

[PASS] git: working tree clean on 'main' (ignoring this notebook)  — clean
[INFO] git: last commit  — 5604a5b yashraj-a11y | Corrected deferred-score claims: five scores plus market size, and README now leads with held-out test metrics
[PASS] git: no tracked file exceeds GitHub's 100 MB limit  — largest: artifacts/fused_features/X_train_fused.npz (14.4 MB)
[PASS] git: .env is not tracked and is git-ignored
[PASS] git: node_modules, dist, mlruns, logs and the FAISS index are ignored  — frontend/node_modules, frontend/dist, mlruns, logs, artifacts/vector_db/faiss.index


[PASS] git: no obvious secrets in tracked text files  — scanned 572 tracked paths
[WARN] git: all commits are pushed to origin/main  — 19 local commit(s) not yet on GitHub (pushing is the owner's decision)


False

## 10 — Result

In [11]:
df = pd.DataFrame(RESULTS, columns=["area", "check", "status", "detail"])
counts = df.status.value_counts().to_dict()
print("PASS", counts.get("PASS", 0), "| WARN", counts.get("WARN", 0), "| INFO", counts.get("INFO", 0), "| FAIL", counts.get("FAIL", 0))
print()
print(df.groupby(["area", "status"]).size().unstack(fill_value=0).to_string())
print("\nWARN / FAIL items:")
for _, r in df[df.status.isin(["WARN", "FAIL"])].iterrows():
    print(f"  [{r.status}] {r.area}: {r.check} — {r.detail[:140]}")
assert counts.get("FAIL", 0) == 0, "release verification found real problems — see FAIL rows above"
print("\nNo FAIL checks. The release is internally consistent.")

PASS 78 | WARN 3 | INFO 5 | FAIL 0



status    INFO  PASS  WARN
area                      
api          1    14     0
assets       1     8     0
claims       0    16     0
code         1     8     1
docs         0     3     1
files        0    19     0
frontend     0     1     0
git          1     5     1
lint         0     1     0
tests        1     3     0

WARN / FAIL items:
  [WARN] docs: placeholders still waiting on the owner — ['[demo video URL]', '[your name]']
  [WARN] code: every mention of a neural network in the docs is a negation / deferral — ['/Users/Yash/AI-Ventutre-Intelligence-Platform/docs/DEMO_GUIDE.md:260: **Q6. "Where\'s the neural network?" / "Where\'s founder strength?"*
  [WARN] git: all commits are pushed to origin/main — 19 local commit(s) not yet on GitHub (pushing is the owner's decision)

No FAIL checks. The release is internally consistent.


### What this notebook did **not** verify

| Not checked | Why / where it stands |
|---|---|
| Docker image builds and runs | No Docker on the development machine; the configuration is written and reviewed, never built |
| CI passes on GitHub | The workflow file exists; it has not been observed running (the work is not pushed) |
| The prose is *correct* | Only that quoted numbers match artifacts and links resolve; wording was reviewed by hand |
| The UI as a person experiences it | Driven and screenshotted with headless Chrome in Phase G (no console errors); not hand-clicked |
| Mermaid diagram syntax | Validated with the real Mermaid parser in Phase C, not in Python |
| Performance on current-day campaigns | Training data ends in 2017 and the split is random — the biggest open question ([Performance Report §12](../docs/PERFORMANCE_REPORT.md)) |
| Licensing of the Kaggle dataset | Not redistributed here; its own terms apply |

### Day 20 in one paragraph

No new ML. The empty README and LICENSE were written; every claim in the docs was tied to an artifact in an
evidence ledger (dropping the ones that could not be proven); the architecture, API, performance, demo,
interview, resume and research documents were produced from the real system; the UI was rendered for the first
time, which found and fixed three bugs; and this notebook re-verifies the whole release. **Roadmap V1.1 is
complete.** The next step is the chronological evaluation in the [Research Roadmap](../docs/RESEARCH_ROADMAP.md).